## 기본 대화

In [1]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-5.6-luna")

from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="너는 미녀와 야수에 나오는 미녀야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 개스톤입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요?"),
]

model.invoke(messages)

AIMessage(content='안녕하세요, 개스톤. 초대는 고맙지만, 오늘 저녁은 사양할게요. 저는 책을 읽으며 조용한 시간을 보내고 싶답니다. 당신도 좋은 저녁 보내세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 61, 'total_tokens': 165, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 47, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EL5fyJ5uEA3otwKQKdwxuxKGmF1xi', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0768e-03c8-74c2-8e6b-81f775a03e30-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 104, 'total_tokens': 165, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token

## StrOutputParser 원하는 부분만 추출

In [2]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

result = model.invoke(messages)
parser.invoke(result)

'안녕하세요, 개스톤. 초대는 고맙지만 오늘 저녁은 사양할게요. 저는 조용히 책을 읽으며 시간을 보내고 싶답니다. 그리고… 저를 잘 알지도 못하면서 서두르지는 말아 주세요.'

## Chain 연산자 ```|``` 를 이용해 더 간단히 줄일 수 있다.

- LCEL은 LangChain에서 여러 컴포넌트를 연결(Chaining)해서 하나의 흐름으로 만드는 방식
- LangChain의 Chaining

In [3]:
chain = model | parser
chain.invoke(messages)

'안녕하세요, 개스톤. 초대해 주셔서 고마워요. 하지만 오늘 저녁은 사양할게요. 저는 책을 읽으며 조용한 시간을 보내고 싶답니다. 게다가 당신은 마을의 모든 아가씨에게 같은 말을 건네는 것 같던데요?'

## 프롬프트 템플릿 이용
- 필요한 부분만 바꿔서 매번 새로 messages 를 만들지 않아도 실행 될 수 있게 하기

In [4]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

result = prompt_template.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

print(result)

messages=[SystemMessage(content='너는 미녀와 야수에 나오는 미녀 역할이다. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕? 저는 야수입니다. 오늘 시간 괜찮으시면 저녁 같이 할까요?', additional_kwargs={}, response_metadata={})]


In [5]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

'안녕하세요, 야수님. 초대해 주셔서 고마워요. 오늘 저녁이라면… 기꺼이 함께할게요. 다만 너무 무서운 표정은 짓지 않기로 약속해 주세요. 책 이야기도 나누면 좋겠어요. 📖'

In [9]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "겨울왕국",
    "character_a": "안나",
    "character_b": "크리스토퍼",
    "activity": "저녁"
})

'안녕, 크리스토퍼! 물론이지! 저녁이라니 정말 좋다! 😊  \n오늘은 내가 맛있는 걸 준비할게—당근도 꼭 챙기고… 스벤 몫도 잊지 말자! 어디서 만날까?'

## LCEL을 이용하여 새로운 체인 만들기
### LCEL과 Pydantic을 이용한 구조화된 출력 체인 만들기
- Pydantic = “데이터 양식 검사기”
- LLM에게 “아무렇게나 말해줘”가 아니라 “이 양식에 맞춰서 결과를 줘”라고 만드는 도구
    - 아래 상황에서는 Adlib에서 answer는 문자열, emotion은 7개 중 하나, intensity는 0.0 ~ 1.0 범위의 숫자라고 미리 양식을 정해둠

In [11]:
from typing import Literal
from pydantic import BaseModel, Field

class Adlib(BaseModel):
    """스토리 설정과 사용자 입력에 반응하는 대사를 만드는 클래스"""
    answer: str = Field(description="스토리 설정과 사용자와의 대화 기록에 따라 생성된 대사")
    main_emotion: Literal["기쁨", "분노", "슬픔", "공포", "냉소", "불쾌", "중립"] = Field(description="대사의 주요 감정")
    main_emotion_intensity: float = Field(description="대사의 주요 감정의 강도 (0.0 ~ 1.0)")

structured_llm = model.with_structured_output(Adlib)
adlib_chain = prompt_template | structured_llm

adlib_chain.invoke({
    "story": "명탐정 코난",
    "character_a": "미란이",
    "character_b": "남도일",
    "activity": "저녁"
})

Adlib(answer='어머, 도일아? 네가 먼저 저녁 먹자고 하다니… 무슨 일 있어? 후훗, 좋아! 오늘 저녁은 내가 같이 먹어줄게. 하지만 또 사건 때문에 중간에 사라지면 안 돼, 알았지?', main_emotion='기쁨', main_emotion_intensity=0.8)